In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PORTFOLIO_VALUE = 100_000_000

project_root = Path.cwd().parent
portfolio_path = project_root / "config" / "portfolio.csv"

portfolio = pd.read_csv(portfolio_path)
portfolio

In [ ]:
required_columns = {'ticker', 'instrument_name', 'asset_class','target_weight','currency','primary_risk_factor'}

#check if any columns are missing

missing_columns = required_columns - set(portfolio.columns)  # converting it to a set for subraction purposes
if missing_columns:
     raise ValueError(f"Missing columns: {missing_columns}")

#check if there are any duplicate tickers
if portfolio["ticker"].duplicated().any():
    raise ValueError("Portfolio contains duplicate tickers.")

#check for missing values
if portfolio.isna().any().any():
    raise ValueError("Portfolio contains missing values.")

#check if there are negative values in target weight
if (portfolio['target_weight'] <= 0).any():
    raise ValueError("target weight cannot be negative or zero.")

#check if the target weights add upto one 
WEIGHT_TOLERANCE = 1e-8
total_weight = portfolio["target_weight"].sum()

if not np.isclose(
    total_weight,
    1.0,
    atol=WEIGHT_TOLERANCE,
    rtol=0.0,
):
    raise ValueError(
        f"Target weights must sum to 1. "
        f"Current total: {total_weight:.10f}"
    )






getting data using yf 

In [ ]:
import yfinance as yf

START_DATE = "2015-01-01"
END_DATE = "2026-08-12"

tickers = portfolio['ticker'].tolist()

raw_data = yf.download(
    tickers=tickers,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=True,
    actions=False,
    progress=False,
)

raw_data.head()

Cleaning the data before finding returns

In [ ]:
prices = raw_data['Close'].copy()
prices  = prices.reindex(columns=tickers)
for ticker in prices.columns:                      # trying to figure out how many prices are missing before we calculate returns we use a for and if  
    missing_count = prices[ticker].isna().sum()    # sstatment to return dates of when the data is missing

    if missing_count > 0:
        missing_dates = prices.index[
            prices[ticker].isna()
        ]

        print(f"{ticker}: {missing_count} missing price(s)")
        print(missing_dates)
        print()            


since we have only 6 missing days of data for over 11 years of market data we can just drop these rows.

calculating returns

In [ ]:
returns = prices.pct_change(fill_method = None)
#print(returns.isna().sum())  #used to see how many na rows are there in each ticker
returns = returns.dropna()
print(returns)

In [ ]:
portfolio["target_market_value"] = (
    portfolio["target_weight"] * PORTFOLIO_VALUE
)

position_values = portfolio.set_index("ticker")[
    "target_market_value"
]

missing_exposures = set(returns.columns) - set(position_values.index)
missing_returns = set(position_values.index) - set(returns.columns)

if missing_exposures:
    raise ValueError(
        f"Returns exist without portfolio exposures: {missing_exposures}"
    )

if missing_returns:
    raise ValueError(
        f"Portfolio positions exist without returns: {missing_returns}"
    )

position_values = position_values.reindex(returns.columns)

position_pnl = returns * position_values
position_pnl.head()

In [ ]:
portfolio_pnl = position_pnl.sum(axis=1)
portfolio_returns = portfolio_pnl/100000000
portfolio_returns .head()

historical var and es calculation

In [ ]:
historical_var = np.percentile(portfolio_returns,1)
#tail_returns  = []
#for daily_returns in portfolio_returns:
#   if daily_returns <= historical_var:
#        tail_returns.append(daily_returns)
#historical_es = np.mean(tail_returns)
tail = portfolio_returns <= historical_var
tail_returns = portfolio_returns[tail]
historical_es = tail_returns.mean()
print(f"Historcial var : {historical_var}")
print(f"Historcial es : {historical_es}")
len(portfolio_returns)
len(tail_returns)
tail_returns.max()
historical_breach_rate = len(tail_returns)/len(portfolio_returns)
print(historical_breach_rate)






next we calculate rollling historical var 

In [ ]:
window_size = 252

forecast_dates = []
var_cutoffs = []
actual_returns = []
breach_results = []

for forecast_index in range(window_size, len(portfolio_returns)):
    historical_window = portfolio_returns.iloc[
        forecast_index - window_size : forecast_index
    ]

    var_cutoff = np.percentile(historical_window, 1)
    actual_return = portfolio_returns.iloc[forecast_index]
    forecast_date = portfolio_returns.index[forecast_index]

    breach = actual_return <= var_cutoff

    forecast_dates.append(forecast_date)
    var_cutoffs.append(var_cutoff)
    actual_returns.append(actual_return)
    breach_results.append(breach)

backtest_results = pd.DataFrame({
    "date": forecast_dates,
    "var_cutoff": var_cutoffs,
    "actual_return": actual_returns,
    "breach": breach_results,
})

backtest_results = backtest_results.set_index("date")
backtest_results.head()

number_of_breaches = backtest_results["breach"].sum()
number_of_forecasts = len(backtest_results)
rollling_breach_rate = number_of_breaches / number_of_forecasts

print(rolling_breach_rate)